# Add Insulation Column to U-Value Table

This notebook adds an "Insulation" column to the u_value_table_to_modify.csv file:
- "yes" for Wall entries with "12 cm insulation" mentioned
- "yes" for Roof entries with "14 cm insulation" mentioned  
- "no" for all other entries

In [ ]:
!uv pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Read the CSV file (using semicolon delimiter)
df = pd.read_csv('/mnt/d/heatpump_data/u_values/u_value_table_to_modify.csv', sep=';')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Shape: (112, 10)

Columns: ['Code_Construction', 'Code_StatusDataset', 'Code_Country', 'Code_ElementType', 'Code_DataType_Construction', 'Code_Construction_ConstructionYearClass', 'Type_Construction', 'Year1_Construction', 'Year2_Construction', 'U']

First few rows:


,Code_Construction,Code_StatusDataset,Code_Country,Code_ElementType,Code_DataType_Construction,Code_Construction_ConstructionYearClass,Type_Construction,Year1_Construction,Year2_Construction,U
0,DE.Ceiling.ReEx.01.01,Typology,DE,Ceiling,ReEx,DE.01,wooden beam ceiling with visible beams,0,1859,1.0
1,DE.Ceiling.ReEx.02.01,Typology,DE,Ceiling,ReEx,DE.02,NaN,1860,1918,1.0
2,DE.Ceiling.ReEx.03.01,Typology,DE,Ceiling,ReEx,DE.03,wooden beam ceiling,1919,1948,0.8
3,DE.Ceiling.ReEx.04.01,Typology,DE,Ceiling,ReEx,DE.04,cavity blocks ceiling,1949,1957,2.1
4,DE.Ceiling.ReEx.05.01,Typology,DE,Ceiling,ReEx,DE.05,NaN,1958,1968,2.1


In [ ]:
# Add Insulation column
# Check for Wall with 12 cm insulation
wall_12cm_mask = (
    (df['Code_ElementType'] == 'Wall') & 
    (df['Type_Construction'].str.contains('12 cm', case=False, na=False))
)

# Check for Roof with 14 cm insulation
roof_14cm_mask = (
    (df['Code_ElementType'] == 'Roof') & 
    (df['Type_Construction'].str.contains('14 cm insulation', case=False, na=False))
)

# Set Insulation column
df['Insulation'] = 'no'
df.loc[wall_12cm_mask | roof_14cm_mask, 'Insulation'] = 'yes'

print(f"Rows with Insulation='yes': {(df['Insulation'] == 'yes').sum()}")
print(f"\nWall entries with 12 cm insulation:")
print(df[wall_12cm_mask][['Code_ElementType', 'Type_Construction', 'Insulation']])
print(f"\nRoof entries with 14 cm insulation:")
print(df[roof_14cm_mask][['Code_ElementType', 'Type_Construction', 'Insulation']])

Rows with Insulation='yes': 2

Wall entries with 12 cm insulation:
   Code_ElementType                            Type_Construction Insulation
81             Wall  masonry with 12 cm render insulation system        yes

Roof entries with 14 cm insulation:
   Code_ElementType                  Type_Construction Insulation
51             Roof  tilted roof with 14 cm insulation        yes


In [ ]:
# Database connection
DATABASE_URL = os.getenv('DATABASE_URL')
if not DATABASE_URL:
    raise ValueError("DATABASE_URL environment variable must be set")

# Connect to database and update Insulation column
conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

try:
    updated_count = 0
    
    # Update each row in the database
    for _, row in df.iterrows():
        code_construction = row['Code_Construction']
        insulation_value = row['Insulation']
        
        update_query = """
        UPDATE u_value_table_with_insulation
        SET "Insulation" = %s
        WHERE "Code_Construction" = %s
        """
        
        cursor.execute(update_query, (insulation_value, code_construction))
        if cursor.rowcount > 0:
            updated_count += 1
    
    conn.commit()
    print(f"✓ Successfully updated {updated_count} rows in database")
    print(f"\nInsulation column summary:")
    print(df['Insulation'].value_counts())
    
except Exception as e:
    conn.rollback()
    print(f"✗ Error updating database: {e}")
    raise
finally:
    cursor.close()
    conn.close()

KeyboardInterrupt: 

In [ ]:
import os
import psycopg2
import pandas as pd

# Read the new Excel file with extra rows
new_file_path = '/mnt/d/heatpump_data/u_values/u_value_table_to_modify_with_insulation_filled.xlsx'
df_new = pd.read_excel(new_file_path)

print(f"New Excel file rows: {len(df_new)}")
print(f"Columns: {df_new.columns.tolist()}")

# Database connection
DATABASE_URL = os.getenv('DATABASE_URL')
if not DATABASE_URL:
    raise ValueError("DATABASE_URL environment variable must be set")

conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

try:
    # Get existing Code_Construction values from database
    cursor.execute('SELECT "Code_Construction" FROM u_value_table_with_insulation')
    existing_codes = set(row[0] for row in cursor.fetchall())
    print(f"\nExisting rows in database: {len(existing_codes)}")
    
    # Find new rows (rows not in database)
    df_new['Code_Construction'] = df_new['Code_Construction'].astype(str)
    new_rows = df_new[~df_new['Code_Construction'].isin(existing_codes)].copy()
    
    print(f"New rows to insert: {len(new_rows)}")
    
    if len(new_rows) > 0:
        print(f"\nNew rows preview:")
        print(new_rows[['Code_Construction', 'Code_ElementType', 'U', 'Insulation']].head(10))
        
        # Prepare data for insertion
        required_columns = [
            'Code_Construction', 'Code_StatusDataset', 'Code_Country',
            'Code_ElementType', 'Code_DataType_Construction',
            'Code_Construction_ConstructionYearClass', 'Type_Construction',
            'Year1_Construction', 'Year2_Construction', 'U', 'Insulation'
        ]
        
        # Ensure all required columns exist
        missing_columns = [col for col in required_columns if col not in new_rows.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        # Replace NaN with None for proper NULL handling
        df_insert = new_rows[required_columns].copy()
        df_insert = df_insert.where(pd.notnull(df_insert), None)
        
        # Insert new rows
        insert_query = """
        INSERT INTO u_value_table_with_insulation (
            "Code_Construction", "Code_StatusDataset", "Code_Country",
            "Code_ElementType", "Code_DataType_Construction",
            "Code_Construction_ConstructionYearClass", "Type_Construction",
            "Year1_Construction", "Year2_Construction", "U", "Insulation"
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        inserted_count = 0
        for _, row in df_insert.iterrows():
            try:
                cursor.execute(insert_query, tuple(row))
                inserted_count += 1
            except psycopg2.IntegrityError as e:
                # Skip if duplicate (shouldn't happen, but just in case)
                print(f"  Skipping duplicate: {row['Code_Construction']}")
                continue
        
        conn.commit()
        print(f"\n✓ Successfully inserted {inserted_count} new rows")
        
        # Verify total count
        cursor.execute('SELECT COUNT(*) FROM u_value_table_with_insulation')
        total_count = cursor.fetchone()[0]
        print(f"✓ Total rows in database: {total_count}")
    else:
        print("\n✓ No new rows to insert. Database is up to date.")
        
except Exception as e:
    conn.rollback()
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()
    raise
finally:
    cursor.close()
    conn.close()

New Excel file rows: 132
Columns: ['Code_Construction', 'Code_StatusDataset', 'Code_Country', 'Code_ElementType', 'Code_DataType_Construction', 'Code_Construction_ConstructionYearClass', 'Type_Construction', 'Year1_Construction', 'Year2_Construction', 'U', 'Insulation']

Existing rows in database: 112
New rows to insert: 20

New rows preview:
        Code_Construction Code_ElementType     U Insulation
11  DE.Ceiling.ReEx.12.01          Ceiling  0.25         no
23     DE.Door.ReEx.12.01             Door  1.80         no
45    DE.Floor.SyAv.10.01            Floor  0.43         no
46    DE.Floor.SyAv.11.01            Floor  0.43         no
47    DE.Floor.SyAv.12.01            Floor  0.43         no
69     DE.Roof.SyAv.10.01             Roof  0.34         no
70     DE.Roof.SyAv.11.01             Roof  0.34         no
71     DE.Roof.SyAv.12.01             Roof  0.34         no
92     DE.Wall.SyAv.02.01             Wall  1.40         no
93     DE.Wall.SyAv.03.01             Wall  1.40       